# Facade - Structural Design Pattern

A `facade` provides a simple, cohesive API over a complex subsystem. It does not add new features, it orchestrates
existing ones so the client code can stay clean and decoupled from low-level details.

```markdown
    Client --> Facade --> [SubsystemA, SubsystemB, SubsystemC]
```

## When to use it

- You have many steps or awkward sequencing to do a common task
- You want to shield your app from churn in third-party/legacy APIs
- You need a stable boundary between layers (application <--> infrastructure)



In [ ]:
# Facade python example

# subsystems

class Amplifier:
    def on(self): print("Amp is on")
    def set_source(self, src: str): print(f"Amp source = {src}")
    def set_volume(self, level: int): print(f"Amp volume = {level}")

class Projector:
    def on(self): print("Projector is on")
    def set_input(self, src: str): print(f"Projector input = {src}")

class Screen:
    def down(self): print("Screen is down")
    def up(self): print("Screen up")

class StreamingPlayer:
    def on(self): print("Player on")
    def play(self, movie: str): print(f"Playing {movie}")
    def off(self): print("Player off")

# Facade

class HomeTheaterFacade:
    def __init__(self, amp: Amplifier, proj: Projector, screen: Screen, player: StreamingPlayer):
        self.amp = amp
        self.proj = proj
        self.screen = screen
        self.player = player
    
    def watch_movie(self, title:str, volume: int=5):
        self.screen.down()
        self.proj.on()
        self.proj.set_input("HDMI")
        self.amp.on()
        self.amp.set_source("HDMI")
        self.amp.set_volume(volume)
        self.player.on()
        self.player.play(title)

    def end_movie(self):
        self.player.off()
        self.screen.up()
        print("Shutdown sequence complete")

if __name__ == "__main__":
    facade = HomeTheaterFacade(Amplifier(), Projector(), Screen(), StreamingPlayer())
    facade.watch_movie("Inception", volume=7)
    facade.end_movie()

## Similar patterns and differences from Facade 

- `Adapter Pattern`: makes one class look like another. The difference is that facade simplifies things under the hood
- `Proxy Pattern`: same interface, adds indirection (access control, lazy load). Facade dofferent interface.
- `Mediator Pattern`: Coordinates peers taht don't know about each other. Facade coordinates subsystems you call.

## Quick practices

1. Wrap Python’s pathlib, zipfile, and shutil into a BackupFacade with:

- backup(src_dir: Path, destination_zip: Path)

- restore(zip_path: Path, target_dir: Path)

2. Create an MLPipelineFacade that chains: load CSV → split → train → evaluate, using scikit‑learn.

In [ ]:
# Quick practice


from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path
from typing import Iterable, List, Optional
import fnmatch
import shutil
import zipfile
import os

class BackupFacade:

    """
     A thin facade over pathlib, zipfile, and shutil to create and restore backups.

    - backup(src_dir, zip_path): creates a ZIP archive from a folder.
    - restore(zip_path, target_dir): extracts a ZIP safely into a folder.
    - list_contents(zip_path): lists archive members.

    """

    exclude_globs: List[str] = field(default_factory= list)
    follow_symlinks: bool = False
    compression: int = zipfile.ZIP_DEFLATED
    compress_level: Optional[int] = None

    def backup(
            self,
            src_dir: Path | str,
            zip_path: Path | str,
            *,
            store_relative_paths: bool = True,
            include_hidden: bool = True,
            overwrite: bool = True
    ) -> Path:
        src = Path(src_dir).resolve()
        if not src.exists() or not src.is_dir():
            raise ValueError(f"Source directory '{src}' does not exist or is not a directory.")
        
        out = Path(zip_path).resolve()